# HRD Feature Engineering — Demo

Demonstrates the four feature transformation classes on **synthetic expression data**
that mimics real HRD/HRP differences.

1. `RankBasedFeatures` — binary gene-pair comparisons
2. `PathwayScores` — ssGSEA / mean-rank pathway activity
3. `ExpressionRatios` — log-ratios of functionally related genes
4. `HRDFeaturePipeline` — chains all of the above

All features are designed to be **platform-agnostic** — they rely on relative
ordering or ratios rather than absolute expression levels.

In [ ]:
import sys, os
import numpy as np
import pandas as pd

# Add project root so imports work from the notebook
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

from v2.feature_engineering.gene_sets import GENE_SETS, HR_REPAIR, ALT_EJ, DDR_BROAD
from v2.feature_engineering.feature_transforms import (
    RankBasedFeatures,
    PathwayScores,
    ExpressionRatios,
    HRDFeaturePipeline,
)

np.random.seed(42)
print('Imports OK')

## 1. Generate Synthetic Expression Data

We create a (genes x samples) matrix with:
- 200 genes covering all curated gene sets + filler genes
- 100 HRD samples and 100 HRP samples
- HRD samples have **lower HR-repair gene expression** and **higher POLQ / alt-EJ**
- Noise is added to make the separation imperfect

In [ ]:
# Collect all known genes from our curated sets
all_pathway_genes = sorted(set(
    g for gl in GENE_SETS.values() for g in gl
))
# Pad to 200 genes with fillers
n_filler = max(0, 200 - len(all_pathway_genes))
filler_genes = [f'FILLER_{i}' for i in range(n_filler)]
gene_names = all_pathway_genes + filler_genes
n_genes = len(gene_names)

n_hrd, n_hrp = 100, 100
n_samples = n_hrd + n_hrp
sample_ids = [f'HRD_{i:03d}' for i in range(n_hrd)] + \
             [f'HRP_{i:03d}' for i in range(n_hrp)]

# Baseline expression (log2 scale, ~5-15 range like RNA-seq TPM)
baseline = np.random.normal(10, 2, size=(n_genes, n_samples))

# Create HRD-specific shifts
hr_idx = [i for i, g in enumerate(gene_names) if g in set(HR_REPAIR)]
altej_idx = [i for i, g in enumerate(gene_names) if g in set(ALT_EJ)]
ddr_idx = [i for i, g in enumerate(gene_names) if g in set(DDR_BROAD)]

# HRD samples (first n_hrd columns): lower HR, higher POLQ/alt-EJ, higher DDR
for idx in hr_idx:
    baseline[idx, :n_hrd] -= np.random.normal(2.0, 0.5, n_hrd)  # downregulated
for idx in altej_idx:
    baseline[idx, :n_hrd] += np.random.normal(1.5, 0.5, n_hrd)  # upregulated
for idx in ddr_idx:
    baseline[idx, :n_hrd] += np.random.normal(1.0, 0.5, n_hrd)  # mildly up

# Clip to realistic range
baseline = np.clip(baseline, 0, 20)

expr_df = pd.DataFrame(baseline, index=gene_names, columns=sample_ids)
labels = pd.Series(
    ['HRD'] * n_hrd + ['HRP'] * n_hrp,
    index=sample_ids,
    name='hrd_status'
)

print(f'Expression matrix: {expr_df.shape[0]} genes x {expr_df.shape[1]} samples')
print(f'Labels: {labels.value_counts().to_dict()}')
print(f'\nGene sets coverage:')
for name, gl in GENE_SETS.items():
    overlap = len(set(gl) & set(gene_names))
    print(f'  {name}: {overlap}/{len(gl)} genes present')

## 2. Rank-Based Gene Pair Features

In [ ]:
rbf = RankBasedFeatures(n_pairs=50, min_delta=0.10)
rank_features = rbf.fit_transform(expr_df, labels, max_pairs=50)

print(f'Rank-pair features: {rank_features.shape}')
print(f'  Samples: {rank_features.shape[0]}')
print(f'  Features: {rank_features.shape[1]}')
print(f'\nTop 10 pairs by |delta|:')
pair_info = rbf.get_pair_info()
print(pair_info.head(10).to_string(index=False))

print(f'\nExample feature distributions (mean in HRD vs HRP):')
for col in rank_features.columns[:5]:
    hrd_mean = rank_features.loc[labels == 'HRD', col].mean()
    hrp_mean = rank_features.loc[labels == 'HRP', col].mean()
    print(f'  {col}: HRD={hrd_mean:.2f}, HRP={hrp_mean:.2f}')

## 3. Pathway Activity Scores

In [ ]:
# Test all three scoring methods
for method in ['ssgsea', 'mean_rank', 'zscore']:
    ps = PathwayScores(method=method)
    scores = ps.transform(expr_df)
    print(f'\n--- {method.upper()} ---')
    print(f'Pathway scores: {scores.shape} (samples x pathways)')
    print(f'Pathways scored: {list(scores.columns)}')
    
    # Show HRD vs HRP means for key pathways
    for pw in ['HR_repair', 'alt_EJ', 'DDR_broad']:
        if pw in scores.columns:
            hrd_m = scores.loc[labels == 'HRD', pw].mean()
            hrp_m = scores.loc[labels == 'HRP', pw].mean()
            print(f'  {pw}: HRD={hrd_m:.3f}, HRP={hrp_m:.3f}, diff={hrd_m-hrp_m:.3f}')

In [ ]:
# HRD-relevant subset
ps = PathwayScores(method='ssgsea')
hrd_scores = ps.compute_hrd_relevant_scores(expr_df)
print(f'HRD-relevant pathway scores: {hrd_scores.shape}')
print(f'Pathways: {list(hrd_scores.columns)}')

## 4. Expression Ratio Features

In [ ]:
er = ExpressionRatios()  # default HRD-relevant ratio pairs
ratio_features = er.transform(expr_df)

print(f'Ratio features: {ratio_features.shape}')
print(f'Columns: {list(ratio_features.columns)}')

print(f'\nHRD vs HRP means for each ratio:')
for col in ratio_features.columns:
    hrd_m = ratio_features.loc[labels == 'HRD', col].mean()
    hrp_m = ratio_features.loc[labels == 'HRP', col].mean()
    print(f'  {col}: HRD={hrd_m:.3f}, HRP={hrp_m:.3f}')

In [ ]:
# Select most discriminative ratios
top_ratios = er.select_top_ratios(expr_df, labels, top_k=5)
print('Top 5 most discriminative ratios (by AUC):')
for name, auc in top_ratios:
    print(f'  {name}: AUC = {auc:.3f}')

## 5. Combined HRD Feature Pipeline

In [ ]:
pipe = HRDFeaturePipeline(
    use_rank_pairs=True,
    use_pathway_scores=True,
    use_ratios=True,
    use_raw_expression=False,
    rank_kwargs={'n_pairs': 30, 'min_delta': 0.10},
    pathway_kwargs={'method': 'mean_rank'},
)
combined = pipe.fit_transform(expr_df, labels)

print(f'Combined feature matrix: {combined.shape}')
print(f'\nFeature type breakdown:')
for prefix, label in [('rp_', 'Rank pairs'), ('pw_', 'Pathway scores'),
                       ('rt_', 'Ratios'), ('raw_', 'Raw expression')]:
    cols = [c for c in combined.columns if c.startswith(prefix)]
    if cols:
        print(f'  {label}: {len(cols)} features')

print(f'\nSample of combined features (first 5 samples, first 8 columns):')
print(combined.iloc[:5, :8].to_string())

In [ ]:
# Pipeline with raw expression included
pipe2 = HRDFeaturePipeline(
    use_rank_pairs=True,
    use_pathway_scores=True,
    use_ratios=True,
    use_raw_expression=True,
    rank_kwargs={'n_pairs': 20, 'min_delta': 0.10},
)
combined2 = pipe2.fit_transform(expr_df, labels)
print(f'With raw expression: {combined2.shape}')
for prefix, label in [('rp_', 'Rank pairs'), ('pw_', 'Pathway scores'),
                       ('rt_', 'Ratios'), ('raw_', 'Raw expression')]:
    cols = [c for c in combined2.columns if c.startswith(prefix)]
    if cols:
        print(f'  {label}: {len(cols)} features')

## 6. Correlation Check Between Feature Types

Verify that different feature types capture partially independent information.

In [ ]:
# Compute mean absolute correlation between feature type blocks
rp_cols = [c for c in combined.columns if c.startswith('rp_')]
pw_cols = [c for c in combined.columns if c.startswith('pw_')]
rt_cols = [c for c in combined.columns if c.startswith('rt_')]

def mean_abs_corr(df, cols_a, cols_b):
    """Mean |correlation| between two blocks of columns."""
    if not cols_a or not cols_b:
        return float('nan')
    block_a = df[cols_a].values
    block_b = df[cols_b].values
    # Correlation matrix between blocks
    corr_matrix = np.corrcoef(block_a.T, block_b.T)
    n_a = len(cols_a)
    # Extract cross-correlation block
    cross = corr_matrix[:n_a, n_a:]
    return np.nanmean(np.abs(cross))

pairs_list = [
    ('Rank pairs', 'Pathway scores', rp_cols, pw_cols),
    ('Rank pairs', 'Ratios', rp_cols, rt_cols),
    ('Pathway scores', 'Ratios', pw_cols, rt_cols),
]

print('Mean |correlation| between feature types:')
for name_a, name_b, ca, cb in pairs_list:
    r = mean_abs_corr(combined, ca, cb)
    print(f'  {name_a} vs {name_b}: {r:.3f}')

print(f'\n(Lower values → more independent information → more value in combining them)')

## 7. Platform Agnosticism Demo

Show that rank-based features are invariant to monotonic transformations
(simulating RNA-seq → microarray platform differences).

In [ ]:
# Simulate microarray by applying a non-linear transform + different noise
expr_microarray = np.log2(2**expr_df + 1)  # compress dynamic range
expr_microarray += np.random.normal(0, 0.3, expr_microarray.shape)  # different noise
expr_microarray = expr_microarray.clip(0, 20)

# Apply the SAME fitted rank-pair model to "microarray" data
rank_micro = rbf.transform(expr_microarray)

# Compare: how similar are the features?
common_cols = rank_features.columns.intersection(rank_micro.columns)
agreement = (rank_features[common_cols] == rank_micro[common_cols]).mean()

print(f'Rank-pair agreement between RNA-seq and simulated microarray:')
print(f'  Mean agreement across all pairs: {agreement.mean():.1%}')
print(f'  Min agreement: {agreement.min():.1%}')
print(f'  Max agreement: {agreement.max():.1%}')
print(f'\nThis demonstrates platform agnosticism: rank-based features are')
print(f'largely invariant to monotonic transformations of expression values.')

## Summary

| Feature Type | Count | Platform Agnostic? | Supervised? |
|---|---|---|---|
| Rank-based gene pairs | up to n*(n-1)/2 | Yes (ordinal) | Yes (needs labels) |
| Pathway activity scores | ~10 pathways | Mostly (rank-based) | No |
| Expression ratios | ~15 curated pairs | Partially (log-ratio) | No |
| Raw expression (optional) | ~40 key genes | No | No |

In [ ]:
print('Feature engineering module ready.')
print(f'\nFinal combined matrix would have {combined.shape[1]} features for {combined.shape[0]} samples.')
print(f'Feature type breakdown: {pipe.feature_summary()}')